In [ ]:
# 🏥 Healthcare No-Show Predictor
## Notebook 1: Exploratory Data Analysis (EDA)

**Goal:** Understand the dataset before building any ML model.

**Dataset:** Medical Appointment No-Shows — 110,527 Brazilian hospital appointments (2016)

**Questions we want to answer:**
- What does the data look like?
- Are there missing values or data quality issues?
- What is the class distribution (show vs no-show)?
- Which features might be most predictive?
- Are there any outliers to handle?

In [ ]:
# ── Imports ──────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Display settings
%matplotlib inline
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)
plt.style.use('seaborn-v0_8')

print("✅ All libraries loaded successfully")
print(f"NumPy   : {np.__version__}")
print(f"Pandas  : {pd.__version__}")

In [ ]:
# ── Load Data ────────────────────────────────────────
df = pd.read_csv('../data/KaggleV2-May-2016.csv')

print("=== DATASET LOADED ===")
print(f"Shape    : {df.shape}")
print(f"Rows     : {df.shape[0]:,}")
print(f"Columns  : {df.shape[1]}")

In [ ]:
# ── First Look ───────────────────────────────────────
print("=== FIRST 5 ROWS ===")
df.head()

In [ ]:
# ── Column Info ──────────────────────────────────────
print("=== COLUMNS AND DATA TYPES ===\n")
for col in df.columns:
    print(f"{col:<25} {str(df[col].dtype):<10} "
          f"unique={df[col].nunique()}")

In [ ]:
# ── Clean Column Names ───────────────────────────────
# Original names are inconsistent — standardize them
df = df.rename(columns={
    'PatientId'        : 'patient_id',
    'AppointmentID'    : 'appointment_id',
    'Gender'           : 'gender',
    'ScheduledDay'     : 'scheduled_day',
    'AppointmentDay'   : 'appointment_day',
    'Age'              : 'age',
    'Neighbourhood'    : 'neighbourhood',
    'Scholarship'      : 'scholarship',
    'Hipertension'     : 'hypertension',
    'Diabetes'         : 'diabetes',
    'Alcoholism'       : 'alcoholism',
    'Handcap'          : 'handicap',
    'SMS_received'     : 'sms_received',
    'No-show'          : 'no_show'
})

print("✅ Columns renamed")
print(df.columns.tolist())

In [ ]:
# ── Missing Values ───────────────────────────────────
print("=== MISSING VALUES ===\n")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    'missing_count'  : missing,
    'missing_percent': missing_pct
})

print(missing_df[missing_df['missing_count'] > 0])
print("\n✅ No missing values!" 
      if missing.sum() == 0 
      else "⚠️ Missing values found!")

In [ ]:
# ── Basic Statistics ─────────────────────────────────
print("=== DESCRIPTIVE STATISTICS ===\n")
df.describe()

In [ ]:
# ── Target Variable: No-Show ─────────────────────────
print("=== TARGET VARIABLE DISTRIBUTION ===\n")

counts = df['no_show'].value_counts()
pcts   = df['no_show'].value_counts(normalize=True) * 100

print(f"Showed up  (No) : {counts['No']:>6,}  ({pcts['No']:.1f}%)")
print(f"No-show   (Yes) : {counts['Yes']:>6,}  ({pcts['Yes']:.1f}%)")
print(f"Total           : {len(df):>6,}")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Bar chart
axes[0].bar(['Showed Up', 'No-Show'],
            [counts['No'], counts['Yes']],
            color=['steelblue', 'tomato'],
            alpha=0.8)
axes[0].set_title('Appointment Attendance Count')
axes[0].set_ylabel('Number of Appointments')
for i, v in enumerate([counts['No'], counts['Yes']]):
    axes[0].text(i, v + 500, f'{v:,}', ha='center')

# Pie chart
axes[1].pie([counts['No'], counts['Yes']],
            labels=['Showed Up', 'No-Show'],
            colors=['steelblue', 'tomato'],
            autopct='%1.1f%%',
            startangle=90)
axes[1].set_title('Attendance Distribution')

plt.suptitle('Target Variable: No-Show Analysis', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/target_distribution.png', 
            dpi=150, bbox_inches='tight')
plt.show()

print("\n💡 Class imbalance observation:")
print(f"   Dataset is {pcts['No']:.0f}/{pcts['Yes']:.0f} "
      f"show/no-show — imbalanced!")

In [ ]:
# ── Age Distribution ─────────────────────────────────
print("=== AGE ANALYSIS ===\n")

# Basic stats
print(f"Mean age   : {df['age'].mean():.1f}")
print(f"Median age : {df['age'].median():.1f}")
print(f"Std dev    : {df['age'].std():.1f}")
print(f"Min age    : {df['age'].min()}")
print(f"Max age    : {df['age'].max()}")

# IQR and outliers ← using what you learned in stats!
Q1  = df['age'].quantile(0.25)
Q3  = df['age'].quantile(0.75)
IQR = Q3 - Q1
lower_fence = Q1 - 1.5 * IQR
upper_fence = Q3 + 1.5 * IQR

outliers = df[(df['age'] < lower_fence) | 
              (df['age'] > upper_fence)]
print(f"\nQ1={Q1}, Q3={Q3}, IQR={IQR}")
print(f"Fences: [{lower_fence:.1f}, {upper_fence:.1f}]")
print(f"Age outliers: {len(outliers)}")
print(f"Negative ages: {len(df[df['age'] < 0])}")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram
axes[0].hist(df['age'], bins=50, 
             color='steelblue', alpha=0.7,
             edgecolor='white')
axes[0].set_title('Age Distribution')
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Count')
axes[0].axvline(df['age'].mean(), 
                color='red', linestyle='--',
                label=f"Mean={df['age'].mean():.1f}")
axes[0].axvline(df['age'].median(), 
                color='green', linestyle='--',
                label=f"Median={df['age'].median():.1f}")
axes[0].legend()

# Box plot ← IQR visualization!
axes[1].boxplot(df['age'], vert=False)
axes[1].set_title('Age Box Plot (IQR + Outliers)')
axes[1].set_xlabel('Age')

plt.tight_layout()
plt.show()

In [ ]:
# ── No-Show by Gender ────────────────────────────────
print("=== NO-SHOW BY GENDER ===\n")

gender_noshow = pd.crosstab(
    df['gender'],
    df['no_show'],
    normalize='index'
) * 100

print(gender_noshow.round(1))

# Plot
gender_noshow.plot(kind='bar',
                   color=['steelblue', 'tomato'],
                   alpha=0.8,
                   figsize=(7, 4))
plt.title('No-Show Rate by Gender')
plt.xlabel('Gender')
plt.ylabel('Percentage %')
plt.xticks(rotation=0)
plt.legend(['Showed Up', 'No-Show'])
plt.tight_layout()
plt.show()

In [ ]:
# ── Medical Conditions vs No-Show ────────────────────
conditions = ['hypertension', 'diabetes', 
              'alcoholism', 'handicap']

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for idx, condition in enumerate(conditions):
    ct = pd.crosstab(
        df[condition],
        df['no_show'],
        normalize='index'
    ) * 100
    
    ct.plot(kind='bar',
            ax=axes[idx],
            color=['steelblue', 'tomato'],
            alpha=0.8)
    axes[idx].set_title(f'No-Show Rate by {condition}')
    axes[idx].set_xlabel(condition)
    axes[idx].set_ylabel('Percentage %')
    axes[idx].set_xticklabels(['No', 'Yes'], rotation=0)
    axes[idx].legend(['Showed Up', 'No-Show'])

plt.suptitle('Medical Conditions vs No-Show Rate',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── SMS Received vs No-Show ──────────────────────────
print("=== SMS IMPACT ON ATTENDANCE ===\n")

sms_noshow = pd.crosstab(
    df['sms_received'],
    df['no_show'],
    normalize='index'
) * 100

print(sms_noshow.round(1))
print("\n💡 Did SMS reminders help reduce no-shows?")

# Plot
sms_noshow.plot(kind='bar',
                color=['steelblue', 'tomato'],
                alpha=0.8,
                figsize=(7, 4))
plt.title('No-Show Rate: SMS Received vs Not')
plt.xlabel('SMS Received (0=No, 1=Yes)')
plt.ylabel('Percentage %')
plt.xticks(rotation=0)
plt.legend(['Showed Up', 'No-Show'])
plt.tight_layout()
plt.show()

In [ ]:
# ── Waiting Days ─────────────────────────────────────
# How many days between scheduling and appointment?
df['scheduled_day']   = pd.to_datetime(df['scheduled_day'])
df['appointment_day'] = pd.to_datetime(df['appointment_day'])

df['waiting_days'] = (df['appointment_day'] - 
                      df['scheduled_day']).dt.days

print("=== WAITING DAYS STATS ===\n")
print(f"Mean   : {df['waiting_days'].mean():.1f} days")
print(f"Median : {df['waiting_days'].median():.1f} days")
print(f"Min    : {df['waiting_days'].min()} days")
print(f"Max    : {df['waiting_days'].max()} days")
print(f"\nNegative waiting days (data error!): "
      f"{len(df[df['waiting_days'] < 0])}")

# Plot waiting days vs no-show
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df['waiting_days'].clip(0, 100),
             bins=50, color='steelblue', alpha=0.7)
axes[0].set_title('Waiting Days Distribution')
axes[0].set_xlabel('Days')
axes[0].set_ylabel('Count')

# Box plot by no-show
df.boxplot(column='waiting_days',
           by='no_show',
           ax=axes[1])
axes[1].set_title('Waiting Days by No-Show')
axes[1].set_xlabel('No-Show')
axes[1].set_ylabel('Waiting Days')

plt.tight_layout()
plt.show()

In [ ]:
# ── Correlation Heatmap ──────────────────────────────
print("=== FEATURE CORRELATIONS ===\n")

# Convert target to binary
df['no_show_binary'] = (df['no_show'] == 'Yes').astype(int)

# Select numeric features
numeric_cols = ['age', 'scholarship', 'hypertension',
                'diabetes', 'alcoholism', 'handicap',
                'sms_received', 'waiting_days',
                'no_show_binary']

corr_matrix = df[numeric_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix,
            annot=True,
            fmt='.2f',
            cmap='coolwarm',
            center=0,
            square=True)
plt.title('Feature Correlation Heatmap', 
          fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n💡 Features most correlated with no_show:")
target_corr = corr_matrix['no_show_binary'].drop('no_show_binary')
print(target_corr.sort_values(ascending=False))

In [ ]:
## 📋 EDA Summary — Key Findings

### Data Quality
- ✅ No missing values
- ⚠️ Negative age values exist → need cleaning
- ⚠️ Negative waiting days exist → need cleaning

### Class Distribution
- 80% showed up, 20% no-show → imbalanced dataset
- Will need to handle imbalance in modeling phase

### Key Observations
1. **SMS reminders** — surprisingly, patients who received 
   SMS had HIGHER no-show rates (needs investigation)
2. **Waiting days** — longer wait = more likely to no-show
3. **Age** — younger patients more likely to no-show
4. **Medical conditions** — hypertension patients more 
   likely to show up (higher health awareness)

### Features for Modeling
Strong candidates:
- waiting_days (engineered feature)
- age
- sms_received
- scholarship
- hypertension

### Next Steps
→ Notebook 02: Data cleaning and preprocessing
→ Handle negative ages and waiting days
→ Encode categorical variables
→ Scale features for ML models

In [ ]:
print("✅ EDA Complete!")
print("Next: 02_preprocessing.ipynb")